# Fake News Detection Using LSTM

This notebook implements a bidirectional LSTM model for binary classification of fake vs. real news articles. The model uses word embeddings and sequence processing to capture semantic patterns in news text.

## Problem Overview

**Objective**: Classify news articles as fake (0) or real (1) based on their title and text content.

**Approach**: Deep learning with bidirectional LSTM to capture sequential dependencies and context in text data.

**Key Challenges**:
- Text preprocessing and normalization
- Sequence length management (padding/truncation)
- Preventing overfitting with regularization
- Generalization to unseen datasets

In [ ]:
# Import required libraries
# Core data processing and ML frameworks
import pandas as pd
import numpy as np
import nltk
import gdown
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import stanza
import joblib
import dateparser
import gzip
import shutil
import pickle
from IPython.display import display, Markdown
from collections import Counter
from itertools import islice
from math import log
from gensim.models import KeyedVectors
from nltk.tokenize import word_tokenize
from scipy.stats import chi2_contingency
from scipy.sparse import vstack, hstack

# Deep learning components for LSTM model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential, save_model, load_model
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout, SpatialDropout1D, BatchNormalization, Bidirectional
from keras.callbacks import EarlyStopping

# Traditional ML tools (for comparison/evaluation)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from wordcloud import WordCloud

# NLP preprocessing tools
nltk.download("stopwords")
from nltk.corpus import stopwords
nltk.download('punkt_tab', quiet = True)
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from collections import Counter


[nltk_data] Downloading package stopwords to /Users/OG1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/OG1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Data Loading

Load the primary training datasets: separate files for fake and real news articles. We'll combine them and add binary labels for classification.

In [ ]:
import pandas as pd
import numpy as np

# Load datasets (same as in eda_aniekan notebook)
fake = pd.read_csv("../data/Fake.csv")
true = pd.read_csv("../data/True.csv")

# Add labels (0 = Fake, 1 = Real)
fake["label"] = 0
true["label"] = 1

# Combine datasets for balanced binary classification
df = pd.concat([fake, true], ignore_index=True)

## Text Preprocessing

# Remove Reuters attribution patterns that may leak source information
# This helps prevent the model from relying on publisher-specific formatting
word_to_remove = '(Reuters)'
df['text'] =  df['text'].str.replace(r'[\(\-–\s]*Reuters[\)\s]*', '', case=False, regex=True)
df['text'] = df['text'].str.replace(r'^[A-Z]+(?:\s+[A-Z]+)*-\s*', '', regex=True)

# Remove stopwords to focus on content-bearing words
# Stopwords provide little discriminative power for fake vs real news
stop_words = stopwords.words("english")
df["title"] = df["title"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words))
df["text"] = df["text"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words))

# Lemmatization: reduce words to root forms (e.g., "running" -> "run")
# This reduces vocabulary size and helps the model generalize across word variations
wnl = WordNetLemmatizer()
df["text"] = df["text"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in x.split()))
df["title"] = df["title"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in x.split()))
print(df.head())
# Drop Subject column (not used in model)
df = df.drop(columns='subject')
df

                                               title  \
0  Donald Trump Sends Out Embarrassing New Year’s...   
1  Drunk Bragging Trump Staffer Started Russian C...   
2  Sheriff David Clarke Becomes An Internet Joke ...   
3  Trump Is So Obsessed He Even Has Obama’s Name ...   
4  Pope Francis Just Called Out Donald Trump Duri...   

                                                text subject  \
0  Donald Trump wish Americans Happy New Year lea...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, revealed former Milwaukee Sheriff D...    News   
3  On Christmas day, Donald Trump announced would...    News   
4  Pope Francis used annual Christmas Day message...    News   

                date  label  
0  December 31, 2017      0  
1  December 31, 2017      0  
2  December 30, 2017      0  
3  December 29, 2017      0  
4  December 25, 2017      0  


,title,text,date,label
0,Donald Trump Sends Out Embarrassing New Year’s...,Donald Trump wish Americans Happy New Year lea...,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian C...,House Intelligence Committee Chairman Devin Nu...,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke ...,"On Friday, revealed former Milwaukee Sheriff D...","December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name ...,"On Christmas day, Donald Trump announced would...","December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Duri...,Pope Francis used annual Christmas Day message...,"December 25, 2017",0
...,...,...,...,...
44893,'Fully committed' NATO back new U.S. approach ...,NATO ally Tuesday welcomed President Donald Tr...,"August 22, 2017",1
44894,LexisNexis withdrew two product Chinese market,"LexisNexis, provider legal, regulatory busines...","August 22, 2017",1
44895,Minsk cultural hub becomes authority,"In shadow disused Soviet-era factory Minsk, st...","August 22, 2017",1
44896,Vatican upbeat possibility Pope Francis visiti...,Vatican Secretary State Cardinal Pietro Paroli...,"August 22, 2017",1


## Model Architecture & Training

### Design Decisions

**Bidirectional LSTM**: Processes sequences in both directions to capture context from past and future tokens, improving understanding of sentence structure and meaning.

**Embedding Layer**: Maps tokenized words to dense 128-dimensional vectors. The embedding learns semantic relationships during training.

**Regularization Strategy**:
- `SpatialDropout1D(0.2)`: Drops entire feature maps (not individual neurons) to prevent overfitting in embeddings
- `Dropout(0.2, recurrent_dropout=0.2)` in LSTM: Regularizes both forward and recurrent connections
- `L2 regularization (0.01)`: Penalizes large weights to prevent overfitting
- `Dropout(0.5)` before final layer: Strong regularization for the dense layer
- `BatchNormalization`: Stabilizes training and can act as a form of regularization

**Sequence Length**: `max_len=80` tokens balances context capture with computational efficiency. Longer sequences provide more context but increase memory and training time.

**Vocabulary Size**: `vocab_size=10000` limits to most frequent words, reducing model complexity while retaining important vocabulary.

In [ ]:
# LSTM model with tokenization

# Combine title and text for richer context
# Titles often contain key information, and combining with text provides full article context
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.regularizers import l2
df['combined_text'] = df['title'].astype(str) + ' ' + df['text'].astype(str)

X_text = df['combined_text'].values
y = df['label'].values

print(f"Number of samples: {len(X_text)}")
print(f"Sample text: {X_text[0][:100]}...")  # show first 100 chars

# Tokenize text: convert words to integer sequences
# OOV token handles words not in vocabulary (important for generalization)
vocab_size = 10000  # maximum number of words to keep
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_text)

# Convert texts to sequences of numbers
X_sequences = tokenizer.texts_to_sequences(X_text)

print(f"Sample sequence: {X_sequences[0][:20]}...")  # show first 20 tokens

# Pad sequences to same length (required for batch processing)
# 'post' padding/truncation: pad at end, truncate from end (preserves beginning context)
max_len = 80
X_padded = pad_sequences(X_sequences, maxlen=max_len, padding='post', truncating='post')

print(f"Padded shape: {X_padded.shape}")

# Train-test split with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Build LSTM model architecture
# Note: Deprecation warnings about input_length/input_shape are cosmetic - model works correctly
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len, input_shape=(max_len,)),
    SpatialDropout1D(0.2),  # Drops entire embedding dimensions to prevent overfitting
    Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(0.01))),
    Dense(8, activation='elu', kernel_regularizer=l2(0.01)),  # ELU helps with gradient flow
    Dropout(0.5),  # Strong dropout before final layer
    BatchNormalization(
        axis=-1, 
        momentum=0.99, 
        epsilon=0.001, 
        center=True, 
        scale=True, 
        beta_initializer='zeros', 
        gamma_initializer='ones',
        moving_mean_initializer='zeros', 
        moving_variance_initializer='ones'  # fixed typo
    ),
    Dense(1, activation='sigmoid')  # Binary classification output
])


# AdamW optimizer with weight decay (L2 regularization) and gradient clipping
# Low learning rate (5e-5) for stable training with regularization
model.compile(
    optimizer=AdamW(learning_rate=5e-5, weight_decay=5e-4, clipnorm=1.0),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nModel Architecture:")
model.summary()

# Training with early stopping to prevent overfitting
# Validation split: 20% of training data used for validation during training
print("\nTraining model...")
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=16,  
    callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
    verbose=1
)

# Save tokenizer for inference on new data
# Critical: must use same tokenizer vocabulary and preprocessing for consistent results
joblib.dump(tokenizer, "tokenizer.pkl")

# Evaluate on held-out test set
print("\nEvaluating on test set...")
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Loss: {loss:.4f}")

# Make predictions and convert probabilities to binary classes
predictions = model.predict(X_test)
pred_classes = (predictions > 0.5).astype(int).flatten()

# Detailed evaluation metrics
print("\n" + "="*50)
print("RESULTS")
print("="*50)
print("\nClassification Report:")
print(classification_report(y_test, pred_classes))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_classes))

# Save model and tokenizer for deployment
model.save('fake_news_lstm_model.keras')
joblib.dump(tokenizer, 'tokenizer.pkl')

print("Model and tokenizer saved successfully!")


Number of samples: 44898
Sample text: Donald Trump Sends Out Embarrassing New Year’s Eve Message; This Disturbing Donald Trump wish Americ...
Sample sequence: [21, 2, 4845, 340, 2688, 17, 3611, 4399, 495, 38, 2758, 21, 2, 1986, 161, 1627, 17, 15, 629, 53]...
Padded shape: (44898, 80)
Training samples: 35918
Test samples: 8980

Model Architecture:


/Users/OG1/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/Users/OG1/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 80, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 80, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         2,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8)              │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,545,265 (5.89 MB)

 Trainable params: 1,545,249 (5.89 MB)

 Non-trainable params: 16 (64.00 B)


Training model...
Epoch 1/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 96s 51ms/step - accuracy: 0.7513 - loss: 3.2520 - val_accuracy: 0.9905 - val_loss: 0.7707
Epoch 2/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 104s 58ms/step - accuracy: 0.9517 - loss: 0.6564 - val_accuracy: 0.9940 - val_loss: 0.2369
Epoch 3/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 114s 64ms/step - accuracy: 0.9629 - loss: 0.2598 - val_accuracy: 0.9942 - val_loss: 0.1031
Epoch 4/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 102s 57ms/step - accuracy: 0.9668 - loss: 0.1563 - val_accuracy: 0.9939 - val_loss: 0.0567
Epoch 5/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 119s 66ms/step - accuracy: 0.9708 - loss: 0.1192 - val_accuracy: 0.9936 - val_loss: 0.0421

Evaluating on test set...
281/281 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.9944 - loss: 0.0419
Test Accuracy: 0.9948
Test Loss: 0.0422
281/281 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step

RESULTS

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      

## Evaluation & Results

### Training Performance

The model achieves very high accuracy (~99.5%) on both validation and test sets. This suggests either:
1. The dataset has strong, learnable patterns
2. Potential overfitting to dataset-specific characteristics
3. Possible data leakage (e.g., source-specific formatting patterns)

**Note**: The validation accuracy (99.36-99.42%) closely matches test accuracy (99.48%), which is positive. However, the extremely high performance warrants caution and testing on truly independent datasets.

### Model Performance Observations

- **Training accuracy**: Increases from 75% to 97% over 5 epochs
- **Validation accuracy**: Stable at ~99.4% across epochs
- **Test accuracy**: 99.48% - matches validation performance
- **Loss**: Decreases from 3.25 to 0.04, indicating strong learning

The model appears to converge well with the regularization strategy preventing severe overfitting during training.

## Testing on External Datasets

To assess generalization, we test the model on two independent datasets that were not used during training. This helps identify potential overfitting or domain shift issues.

In [ ]:

# Load external dataset 1: Fake_Real_News_Data.csv
# This dataset was not used during training - tests true generalization
new_fake_and_true = pd.read_csv("../data/Fake_Real_News_Data.csv")

if 'Unnamed: 0' in new_fake_and_true.columns:
    new_fake_and_true = new_fake_and_true.drop(columns='Unnamed: 0')

new_fake_and_true.head()

# Map labels to match training data format (0 = Fake, 1 = Real)
label_map = {'FAKE': 0, 'REAL': 1}
new_fake_and_true['label'] = new_fake_and_true['label'].map(label_map)

# Apply same preprocessing pipeline as training data
# Critical: must match preprocessing to ensure fair evaluation
stop_words = stopwords.words("english")
new_fake_and_true["title"] = new_fake_and_true["title"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words))
new_fake_and_true["text"] = new_fake_and_true["text"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words))

# Lemmatization
wnl = WordNetLemmatizer()
new_fake_and_true["text"] = new_fake_and_true["text"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in x.split()))
new_fake_and_true["title"] = new_fake_and_true["title"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in x.split()))
print(new_fake_and_true.head())
new_fake_and_true

                                               title  \
0  A whirlwind day D.C. showcase Trump’s unorthod...   
1  In Baltimore's call federal police probe, new ...   
2  Trump Proudly Declares: Most Of The People I’v...   
3  Inside Trump-Bush melodrama: Decades tension d...   
4               Shutdown clash return force December   

                                                text  label  
0  Donald Trump endorsed unabashedly nonintervent...      1  
1  While Justice Department investigation adversa...      1  
2  Trump Proudly Declares: Most Of The People I’v...      0  
3  Donald Trump spent day January 2014 hobnobbing...      1  
4  Notable name include Ray Washburne (Commerce),...      1  


,title,text,label
0,A whirlwind day D.C. showcase Trump’s unorthod...,Donald Trump endorsed unabashedly nonintervent...,1
1,"In Baltimore's call federal police probe, new ...",While Justice Department investigation adversa...,1
2,Trump Proudly Declares: Most Of The People I’v...,Trump Proudly Declares: Most Of The People I’v...,0
3,Inside Trump-Bush melodrama: Decades tension d...,Donald Trump spent day January 2014 hobnobbing...,1
4,Shutdown clash return force December,"Notable name include Ray Washburne (Commerce),...",1
...,...,...,...
6330,Obama To Limit Police Acquisition Of Some Mili...,Obama To Limit Police Acquisition Of Some Mili...,1
6331,EU using taxpayer money give Muslim invader Tu...,BNI Store Oct 29 2016 EU using taxpayer money ...,0
6332,Watching These 55 ISIS Terrorists Get Blown Sm...,Next Story → Judge Judy LOSES IT Hood Rat: “Yo...,0
6333,America’s Streets Will Run With Blood- Mike Adams,America’s Streets Will Run With Blood- Mike Ad...,0


In [ ]:
# Load external dataset 2: WELFake_Dataset.csv
# Another independent dataset to test generalization
welfake_data = pd.read_csv("../data/WELFake_Dataset.csv")
welfake_data.head()

if "Unnamed: 0" in welfake_data.columns:
    welfake_data = welfake_data.drop(columns='Unnamed: 0')
    
# Preprocessing with null handling (this dataset may have missing values)
stop_words = stopwords.words("english")
welfake_data["title"] = welfake_data["title"].apply(
    lambda x: " ".join(word for word in str(x).split() if word not in stop_words) if pd.notna(x) else "")
welfake_data["text"] = welfake_data["text"].apply(
    lambda x: " ".join(word for word in str(x).split() if word not in stop_words) if pd.notna(x) else "")

# Lemmatization with null handling
wnl = WordNetLemmatizer()
welfake_data["text"] = welfake_data["text"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in str(x).split()) if pd.notna(x) else "")
welfake_data["title"] = welfake_data["title"].apply(lambda x: " ".join(wnl.lemmatize(word) for word in str(x).split()) if pd.notna(x) else "")
print(welfake_data.head())
welfake_data.head()
welfake_data.tail()

                                               title  \
0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1                                                      
2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3  Bobby Jindal, raised Hindu, us story Christian...   
4  SATAN 2: Russia unvelis image terrifying new ‘...   

                                                text  label  
0  No comment expected Barack Obama Members #FYF9...      1  
1                     Did post vote Hillary already?      1  
2  Now, demonstrator gathered last night exercisi...      1  
3  A dozen politically active pastor came private...      0  
4  The RS-28 Sarmat missile, dubbed Satan 2, repl...      1  


,title,text,label
72129,Russians steal research Trump hack U.S. Democr...,WASHINGTON (Reuters) - Hackers believed workin...,0
72130,WATCH: Giuliani Demands That Democrats Apologi...,"You know, fantasyland Republicans never questi...",1
72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0
72132,Trump tussle give unpopular Mexican leader muc...,MEXICO CITY (Reuters) - Donald Trump’s combati...,0
72133,Goldman Sachs Endorses Hillary Clinton For Pre...,Goldman Sachs Endorses Hillary Clinton For Pre...,1


### Test on External Dataset 1: Fake_Real_News_Data

Evaluate model performance on the first external dataset using the same tokenizer and preprocessing pipeline.

In [ ]:
# Testing the LSTM Model on external dataset 1

# Combine title and text (same format as training)
dataset1 = (new_fake_and_true['title'] + ' ' + new_fake_and_true['text']).astype(str).tolist()

# Load the original tokenizer (must use same vocabulary from training)
tokenizer = joblib.load("tokenizer.pkl")

# Convert text to sequences using the training vocabulary
# Words not in vocabulary will be mapped to OOV token
sequences = tokenizer.texts_to_sequences(dataset1)

# Pad sequences to match training sequence length
X_new = pad_sequences(sequences, maxlen=80, padding="post")

y_pred = new_fake_and_true['label'].values

# Make predictions
y_new_prob = model.predict(X_new)
predictions = (y_new_prob > 0.5).astype(int).flatten()

# Evaluate performance
print(f"Accuracy score on new dataset: ", accuracy_score(predictions, y_pred))
# print(classification_report(y_pred, y_new_pred))

198/198 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
Accuracy score on new dataset:  0.5810576164167325


**Result**: Accuracy drops to **58.1%** on this external dataset, significantly below the 99.5% test accuracy.

**Interpretation**: This performance gap suggests:
- **Domain shift**: The external dataset may have different writing styles, topics, or sources
- **Overfitting to training data characteristics**: The model may have learned dataset-specific patterns rather than generalizable features
- **Vocabulary mismatch**: Many words in the external dataset may be OOV (out-of-vocabulary), reducing model effectiveness
- **Different preprocessing**: Subtle differences in how the external data was collected/preprocessed

This highlights the importance of testing on truly independent datasets to assess real-world performance.

### Test on External Dataset 2: WELFake_Dataset

Evaluate model performance on a second independent dataset to further assess generalization.

In [ ]:
# Testing on external dataset 2: WELFake_Dataset
dataset2 = (welfake_data['title'] + ' ' + welfake_data['text']).astype(str).tolist()

# Load original tokenizer from model training
tokenizer = joblib.load("tokenizer.pkl")
sequences = tokenizer.texts_to_sequences(dataset2)

# Pad sequences to match training format
X_original = pad_sequences(sequences, maxlen=80, padding="post")

y_original = welfake_data['label'].values

# Make predictions
predictions = model.predict(X_original)
modified_predictions = (predictions > 0.5).astype(int).flatten()

# Evaluate performance
print(f"Accuracy score on dataset2: ", accuracy_score(modified_predictions, y_original))

2255/2255 ━━━━━━━━━━━━━━━━━━━━ 29s 13ms/step
Accuracy score on dataset2:  0.18955000415892645


**Result**: Accuracy drops to **18.96%** on this external dataset, performing worse than random guessing (50% for binary classification).

**Interpretation**: This severe performance degradation indicates:
- **Strong domain shift**: The WELFake dataset likely has very different characteristics from the training data
- **Possible label mismatch**: The label encoding in this dataset may differ from expected (0/1 mapping)
- **Data quality issues**: Missing values or different text formats may affect preprocessing
- **Vocabulary gap**: Significant OOV words reducing model effectiveness

The model's high performance on the original test set does not generalize to these external datasets, highlighting the challenge of domain adaptation in fake news detection.

## Limitations & Next Steps

### Key Limitations Identified

1. **Generalization Gap**: While the model achieves 99.5% accuracy on the original test set, performance drops significantly (58% and 19%) on external datasets, indicating potential overfitting or domain shift.

2. **Vocabulary Constraints**: The fixed vocabulary size (10,000 words) may miss important domain-specific terms in new datasets, leading to many OOV tokens.

3. **Sequence Length**: Fixed `max_len=80` may truncate important context in longer articles, or waste capacity on shorter ones.

4. **Dataset-Specific Patterns**: The model may have learned patterns specific to the training data sources rather than generalizable fake news indicators.

### Recommended Next Steps

1. **Domain Adaptation**: Fine-tune the model on samples from target domains or use domain adaptation techniques.

2. **Vocabulary Expansion**: Increase vocabulary size or use subword tokenization (e.g., SentencePiece, BPE) to handle OOV words better.

3. **Ensemble Methods**: Combine predictions from multiple models trained on different datasets to improve robustness.

4. **Feature Engineering**: Add metadata features (e.g., source credibility, publication date, author information) if available.

5. **Transfer Learning**: Use pre-trained language models (e.g., BERT, RoBERTa) that have learned general language representations.

6. **Data Augmentation**: Augment training data with techniques like back-translation or synonym replacement to improve generalization.

7. **Cross-Dataset Validation**: Implement cross-validation across multiple datasets during training to ensure robustness.

8. **Error Analysis**: Analyze misclassified examples to understand failure modes and improve the model.